# Load Data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
warnings.filterwarnings('ignore')

In [ ]:
# ======================================================================
# STEP 1: LOAD AND PREPARE DATA (REFERENCE + MULTI-KIT MONORAIL)
# ======================================================================

def load_data(model_path, monorail_paths):
    """
    Load and prepare reference (model) data and Monorail data (one or more kits),
    align common columns, and return a single combined DataFrame.

    Parameters
    ----------
    model_path : str
        Path to model.csv (reference experimental campaign).
    monorail_paths : str or list of str
        Path or list of paths to Monorail TestBrakefinal_data_kitXX.csv files.

    Returns
    -------
    df_base : pandas.DataFrame
        Combined DataFrame with:
        - aligned common columns between reference and Monorail,
        - binary label (0/1) where available,
        - 'Source' column (kit ID or 0 for reference),
        - 'DataSource' column (0 = reference, 1 = Monorail).
    """

    # --------------------------------------------------------------
    # Helper: load and clean ONE Monorail file
    # --------------------------------------------------------------
    def load_Monorail(filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)

        # Keep only standard braking
        if 'Non_Standard_Braking' in df.columns:
            df = df[df['Non_Standard_Braking'] == 0]

        # Extract numeric kit ID from filename, e.g. "TestBrakefinal_data_kit06.csv" -> 6
        match = re.search(r'kit(\d+)', os.path.basename(filepath))
        source = int(match.group(1)) if match else -1
        df['Source'] = source

        # Convert "xx sec" string columns to float seconds where possible
        for col in df.select_dtypes(include='object'):
            try:
                df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
            except (AttributeError, ValueError):
                # AttributeError if column is not string-like; ValueError if some values cannot be cast
                continue

        return df

    # --------------------------------------------------------------
    # 1) REFERENCE DATA: load, label, aggregate
    # --------------------------------------------------------------
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

    # Binary label from malfunction code
    leakage_codes = ['C', 'D', 'E', 'F', 'G']
    df_reference['LeakageLabel'] = np.where(
        df_reference['Malfunction'].isin(leakage_codes),
        'Combined leakage',
        'Healthy'
    )

    # Add Source = 0 for reference campaign
    df_reference['Source'] = 0

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay':      ['Brake_timing_delay_exp',      'Release_timing_delay_exp'],
        'Total_energy_delay':      ['Brake_energy_delay_exp',      'Release_energy_delay_exp'],
        'Total_power_delay':       ['Brake_power_delay_exp',       'Release_power_delay_exp'],
        'Total_power_efficiency':  ['Brake_power_efficiency_exp',  'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp',   'Release_energy_efficiency_exp']
    }

    for new_col, (c1, c2) in delay_eff_map.items():
        # If any of these columns are missing in some version of model.csv, guard with .get
        if c1 in df_reference.columns and c2 in df_reference.columns:
            df_reference[new_col] = df_reference[c1] + df_reference[c2]

    # Drop original per-phase columns (only those that actually exist)
    cols_to_drop = [c for pair in delay_eff_map.values() for c in pair if c in df_reference.columns]
    df_reference.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # Rename to your canonical names
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp':  'Buildup_end_pressure_delay',
        'Weight':                          'WV_MeanPressure',
        'Brake_action':                    'EmergencyBrake_action'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # --------------------------------------------------------------
    # 2) MONORAIL DATA: load one or more kit files
    # --------------------------------------------------------------
    if isinstance(monorail_paths, str):
        monorail_paths = [monorail_paths]

    dfs_mono = [load_Monorail(fp) for fp in monorail_paths]
    df_data = pd.concat(dfs_mono, ignore_index=True)

    # --------------------------------------------------------------
    # 3) ALIGN STRUCTURES AND COMBINE
    # --------------------------------------------------------------
    # Ensure 'Source' is integer in both
    df_reference['Source'] = df_reference['Source'].astype(int)
    df_data['Source']      = df_data['Source'].astype(int)

    # Columns common to BOTH datasets
    common_cols = df_reference.columns.intersection(df_data.columns).tolist()

    # Subsets with only common columns + a DataSource flag
    df_reference_subset = df_reference[common_cols].copy()
    df_reference_subset['DataSource'] = 0  # 0 = reference campaign

    df_data_subset = df_data[common_cols].copy()
    df_data_subset['DataSource'] = 1       # 1 = Monorail (real-time) data

    # Stack reference + Monorail
    df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

    # Encode final label column (will be NaN for Monorail if it has no LeakageLabel)
    if 'LeakageLabel' in df_combined.columns:
        df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
        df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Combined leakage': 1})

    # Convert any remaining "xx sec" string columns to float (esp. from model.csv)
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except (AttributeError, ValueError):
            continue

    df_base = df_combined.copy()
    return df_base

model_path = 'model.csv'
monorail_paths = [
    'TestBrakefinal_data_kit01.csv',
    'TestBrakefinal_data_kit06.csv',
    'TestBrakefinal_data_kit27.csv'
]

df = load_data(model_path, monorail_paths)

print(df.shape)
print(df['DataSource'].value_counts(dropna=False))
print(df['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail


In [ ]:
# ============================================================================
# STEP 2: IMPUTE + FEATURE SCALING
# ============================================================================

from sklearn.impute import SimpleImputer

def scale_features(X_train, X_test, X_train_healthy):
    """
    Impute missing values by median, then standardize features.
    Returns imputed+scaled arrays, plus fitted scaler and imputer.
    """
    # 1) Median imputation (fit only on training set)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)
    X_train_healthy_imp = imputer.transform(X_train_healthy)

    # 2) Standardization (fit only on imputed training set)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    X_train_healthy_scaled = scaler.transform(X_train_healthy_imp)

    return X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer

[X_train_scaled_tpe, X_test_scaled_tpe, X_train_healthy_scaled_tpe, scaler, imputer] = scale_features(X_train_tpe, X_test_tpe, X_train_healthy_tpe)

## Common Setup and Shared Helper Function

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# -----------------------------
# 0) label encode helper
# -----------------------------
def make_binary_label(df, label_col="LeakageLabel"):
    """
    Converts LeakageLabel to binary y:
    Healthy -> 0, Fault/Leakage -> 1
    """
    y_raw = df[label_col]
    if y_raw.dtype == object:
        y = (y_raw != "Healthy").astype(int)
    else:
        y = y_raw.astype(int)
    return y

# -----------------------------
# 1) feature / meta split helper
# -----------------------------
def split_features(df, label_col="LeakageLabel", meta_cols=None):
    if meta_cols is None:
        meta_cols = []

    drop_cols = [label_col] + meta_cols
    X = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()
    return X

# -----------------------------
# 2) base supervised pipeline
# -----------------------------
def get_supervised_pipeline(model="xgb"):
    """
    Standard supervised pipeline:
    Impute median -> scale -> classifier
    """
    if model == "xgb":
        clf = XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=42,
            eval_metric="logloss"
        )
    else:
        clf = RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

    pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", clf)
    ])
    return pipe

# -----------------------------
# 3) CV metrics dict
# -----------------------------
scoring = {
    "precision": make_scorer(precision_score, zero_division=0),
    "recall":    make_scorer(recall_score, zero_division=0),
    "f1":        make_scorer(f1_score, zero_division=0),
    "roc_auc":   "roc_auc"
}


## Setup from Code Supervised model

In [ ]:
# Replace only the model with the best estimator
tuned_models = trained_models.copy()
tuned_models['Logistic Regression (Weighted)']['model'] = best_lr
tuned_models['Logistic Regression (SMOTE)']['model'] = best_lr_smote
tuned_models['KNN (Imbalanced)']['model'] = best_knn
tuned_models['Decision Tree (Weighted)']['model'] = best_dt
tuned_models['XGBoost (Weighted)']['model'] = best_xgb_est_tpe
tuned_models['Random Forest (Weighted)']['model'] = best_rf_est_tpe

# Evaluate
cv_results_1_feat_tuned = cv_metrics_table(
    tuned_models, X_train_scaled_1, y_train
)
cv_results_1_feat_tuned

In [ ]:
# Useful Code might use later
# --- IGNORE ---
# Calibrated Probabilities
# from top 4 model
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, precision_recall_curve
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.calibration import CalibratedClassifierCV

# ============================================================
# 1. SELECT TOP-4 MODELS FROM TUNED CV RESULTS (STILL THR=0.5)
# ============================================================

top4_cv = cv_results_1_feat_tuned.sort_values(
    by="F1-Score", ascending=False
).head(4)

print("Top-4 models on training CV (thr=0.5):")
print(top4_cv[["Model", "Type", "F1-Score"]])


# ============================================================
# SOLUTION 2: CALIBRATED PROBABILITIES (ISOTONIC)
# ============================================================

test_rows = []
final_models = {}
final_thresholds = {}

for _, row in top4_cv.iterrows():
    model_name = row["Model"]
    mtype      = row["Type"]

    print(f"\n=== Final training + test evaluation (S2) for: {model_name} ===")

    base_model = tuned_models[model_name]["model"]

    if mtype == "supervised":
        base_estimator = clone(base_model)
    elif mtype == "supervised_smote":
        base_estimator = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', clone(base_model))
        ])
    else:
        print(f"Skipping {model_name} (type={mtype}) – not supervised.")
        continue

    calibrated = CalibratedClassifierCV(
        base_estimator, method='sigmoid', cv=3
    )
    calibrated.fit(X_train_scaled_1, y_train)

    proba_train = calibrated.predict_proba(X_train_scaled_1)[:, 1]

    precision_arr, recall_arr, thresholds = precision_recall_curve(
        y_train, proba_train
    )

    if thresholds.size == 0:
        best_thr = 0.5
        best_f1  = 0.0
        print("  -> thresholds array empty, fallback to 0.5")
    else:
        thr_candidates  = thresholds
        prec_candidates = precision_arr[1:]
        rec_candidates  = recall_arr[1:]

        f1_candidates = 2 * prec_candidates * rec_candidates / (
            prec_candidates + rec_candidates + 1e-8
        )

        best_idx = np.argmax(f1_candidates)
        best_thr = thr_candidates[best_idx]
        best_f1  = f1_candidates[best_idx]

    print(f"  -> Best threshold on TRAIN (calibrated): {best_thr:.3f} "
          f"(F1_train_max = {best_f1:.3f})")

    proba_test = calibrated.predict_proba(X_test_scaled_1)[:, 1]
    y_pred_test = (proba_test >= best_thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
    prec_test = precision_score(y_test, y_pred_test, zero_division=0)
    rec_test  = recall_score(y_test, y_pred_test, zero_division=0)
    f1_test   = f1_score(y_test, y_pred_test, zero_division=0)
    roc_test  = roc_auc_score(y_test, proba_test)

    test_rows.append({
        "Model": model_name,
        "Type": mtype,
        "BestThreshold_train": best_thr,
        "Precision_Test": prec_test,
        "Recall_Test": rec_test,
        "F1_Test": f1_test,
        "ROC-AUC_Test": roc_test,
        "TP_Test": tp,
        "FP_Test": fp,
        "FN_Test": fn,
        "TN_Test": tn,
    })

    final_models[model_name] = calibrated
    final_thresholds[model_name] = best_thr

test_results_top4_S2 = pd.DataFrame(test_rows).sort_values(
    by="F1_Test", ascending=False
).reset_index(drop=True)

print("\n\n=== Top-4 tuned models – TEST performance (Solution 2) ===")
print(test_results_top4_S2)
